# Flood Mapping and Vulnerability Assessment in River and Lagoon Environments

*by Daniele Petrini & Federico Urbini*

## Necessary Imports

In [1]:
# Import libraries and project modules.

import ee
import geemap
import pandas as pd
from pathlib import Path
import sys
import importlib
from IPython.display import display

from utils import config as cfg
from utils import utils_setup as us
from utils import utils_SAR as su
from utils import utils_optical as ou
from utils import utils_final as uf

importlib.reload(cfg)
importlib.reload(us)
importlib.reload(uf)

/home/federico/miniconda3/envs/floods/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


<module 'utils.utils_final' from '/home/federico/Desktop/floods-SAR/utils/utils_final.py'>

## Authentication and Folders

In [2]:
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="floods-2024")
print("Earth Engine initialized correctly.")

Earth Engine initialized correctly.


In [3]:
# Create output folders
Path("outputs").mkdir(exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/maps").mkdir(parents=True, exist_ok=True)
Path("outputs/final").mkdir(parents=True, exist_ok=True)

## Flags

In [4]:
# Define final risk notebook flags.

MONTH = "sept"  # Options: "sept", "oct"
HAZARD_SOURCE = "fusion"  # Options: "fusion", "sar"

RECOMPUTE_INTERMEDIATE_RESULTS = False
MAKE_INTERMEDIATE_MAPS = False
MAKE_FINAL_MAP = True

FINAL_OUTPUT_DIR = Path("outputs") / "tables"
FINAL_MAP_DIR = Path("outputs") / "final"

## Load Configuration and Auxiliary Dataset

In [5]:
# Load project configuration and auxiliary datasets.

project = cfg.get_project_config()

aoi_emilia_romagna = project["aoi_emilia_romagna"]
places = project["places"]
dates = project["dates"]
events = project["events"]

auxiliary = us.load_auxiliary_datasets(aoi_emilia_romagna)

water_occurrence = auxiliary["water_occurrence"]
permanent_water = auxiliary["permanent_water"]
worldcover = auxiliary["worldcover"]
dem = auxiliary["dem"]
slope = auxiliary["slope"]
rivers_image = auxiliary["rivers_image"]

risk_vis = uf.get_risk_vis_params()

print("Configuration and auxiliary datasets loaded.")

Configuration and auxiliary datasets loaded.


In [ ]:
# Build or retrieve the hazard input from SAR confidence or SAR-optical fusion.

if RECOMPUTE_INTERMEDIATE_RESULTS or "results" not in globals():
    sar_vis = su.get_sar_vis_params()

    results = su.run_sar_event(
        event_key=MONTH,
        event=events[MONTH],
        dates=dates,
        permanent_water=permanent_water,
        slope=slope,
        rivers_image=rivers_image,
        places=places,
        vis=sar_vis,
        output_dir=FINAL_OUTPUT_DIR,
        make_map=MAKE_INTERMEDIATE_MAPS,
        **cfg.SAR_PARAMS[MONTH],
    )

    print("SAR results computed.")
else:
    print("Using existing SAR results from current kernel.")

if HAZARD_SOURCE == "sar":
    hazard_input = results["sar_confidence"]

elif HAZARD_SOURCE == "fusion":
    if RECOMPUTE_INTERMEDIATE_RESULTS or "optical_results" not in globals():
        opt_vis = ou.get_optical_vis_params()

        optical_results = ou.run_optical_event(
            event_key=MONTH,
            event=events[MONTH],
            dates=dates,
            sar_confidence=results["sar_confidence"],
            permanent_water=permanent_water,
            places=places,
            output_dir=FINAL_OUTPUT_DIR,
            make_map=MAKE_INTERMEDIATE_MAPS,
            vis=opt_vis,
            **cfg.OPTICAL_PARAMS[MONTH],
        )

        print("Optical/fusion results computed.")
    else:
        print("Using existing optical/fusion results from current kernel.")

    hazard_input = optical_results["fusion_map"]

else:
    raise ValueError("HAZARD_SOURCE must be either 'fusion' or 'sar'.")

print("Selected hazard source:", HAZARD_SOURCE)

SAR results computed.


EEException: Computation timed out.

## Risk Analysis

In [ ]:
# Run vulnerability, exposure, and flood risk analysis.

risk_results = uf.run_final_risk_event(
    event_key=MONTH,
    event=events[MONTH],
    hazard_input=hazard_input,
    hazard_source=HAZARD_SOURCE,
    dem=dem,
    slope=slope,
    worldcover=worldcover,
    water_occurrence=water_occurrence,
    places=places,
    output_dir=FINAL_OUTPUT_DIR,
    make_map=MAKE_FINAL_MAP,
    vis=risk_vis,
    **cfg.RISK_PARAMS[MONTH],
)

print("Low elevation threshold:", risk_results["low_elevation_threshold"].getInfo())

display(risk_results["df_exposure"])
display(risk_results["df_vulnerability_area"])
display(risk_results["df_risk_area"])
display(risk_results["df_roi"])

EEException: Too many concurrent aggregations.

In [ ]:
# Display and save final risk map as an interactive HTML file.

display(risk_results["map"])

map_path = FINAL_MAP_DIR / f"{MONTH}_final_risk_map.html"
risk_results["map"].to_html(str(map_path))
print(f"Final risk map saved to: {map_path}")

## Final Results

In [ ]:
# Save compact final summary tables.

risk_results["df_exposure"].to_csv(
    FINAL_OUTPUT_DIR / f"{MONTH}_final_exposure_summary.csv",
    index=False,
)

risk_results["df_risk_area"].to_csv(
    FINAL_OUTPUT_DIR / f"{MONTH}_final_risk_summary.csv",
    index=False,
)

risk_results["df_roi"].to_csv(
    FINAL_OUTPUT_DIR / f"{MONTH}_final_roi_summary.csv",
    index=False,
)

print("Final summary tables saved.")